<header>
   <p style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
        Getting Started with Content-Based Collection
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

## Introduction

This notebook demonstrates how to work with **Content-Based Collection**

Collections provide a stateless API for managing vector stores, making it easier to create, search, and manage vector embeddings without maintaining persistent connections.

### What You'll Learn

1. Setting up your environment and connecting to Vantage
2. Initializing embedding and chat models
3. Creating a content-based collection from datasets
4. Retrieving collection details and metadata
5. Updating collections (adding/deleting datasets, modifying parameters)
6. Performing similarity search and asking questions
7. Cleaning up resources

---
## 1. Setup

### Install teradatagenai

First, ensure you have the latest version of teradatagenai installed.

In [ ]:
# %pip install teradatagenai --upgrade

### Import Required Modules

In [6]:
import os
import time
from getpass import getpass

from teradatagenai import Collection, TeradataAI, load_data, ContentBasedIndex, ColumnInfo, CollectionManager
from teradatagenai.vector_store.data_classes import HNSW
from teradatagenai.common.constants import CollectionType
from teradataml import create_context, DataFrame, in_schema, remove_context
from teradatasqlalchemy.types import VARCHAR

### Connect to Vantage

**Important Note**: Collection methods are stateless and do not require a database connection for most operations. However, when working with teradataml DataFrames, loading example data, or using certain utility functions, a connection to Vantage is required.

You'll be prompted to provide connection details.

In [ ]:
os.environ['TD_HOST'] = getpass(prompt='hostname: ')
os.environ['TD_USERNAME'] = getpass(prompt='username: ')
os.environ['TD_PASSWORD'] = getpass(prompt='password: ')
os.environ['TD_BASE_URL'] = getpass(prompt='base_url: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'
context = create_context()
print("✓ Connected to Vantage")

Authentication token is generated and set for the session.
✓ Connected to Vantage


### Load Example Data

We'll use the Amazon reviews dataset for this demonstration.

In [ ]:
# Load example datasets
load_data('byom', 'amazon_reviews_25')
load_data('byom', 'amazon_reviews_10')

# Create DataFrame for the main dataset
input_data = DataFrame(in_schema('vs_user', 'amazon_reviews_25'))
print("✓ Example data loaded")
input_data.head(1)

### CHECK AVAILABLE MODELS

In [7]:
CollectionManager.list_available_models()

embedding_models:
                             model_id                      model_name provider  status
0     amazon.titan-embed-text-v1:2:8k      Titan Embeddings G1 - Text   Amazon  ACTIVE
1        amazon.titan-embed-text-v2:0        Titan Text Embeddings V2   Amazon  ACTIVE
2       amazon.titan-embed-image-v1:0  Titan Multimodal Embeddings G1   Amazon  ACTIVE
3         amazon.titan-embed-image-v1  Titan Multimodal Embeddings G1   Amazon  ACTIVE
4             cohere.embed-english-v3                   Embed English   Cohere  ACTIVE
5  cohere.embed-multilingual-v3:0:512              Embed Multilingual   Cohere  ACTIVE
6       cohere.embed-english-v3:0:512                   Embed English   Cohere  ACTIVE
7          amazon.titan-embed-text-v1      Titan Embeddings G1 - Text   Amazon  ACTIVE
8       amazon.titan-embed-g1-text-02        Titan Text Embeddings v2   Amazon  ACTIVE
9                   cohere.embed-v4:0                        Embed v4   Cohere  ACTIVE

chat_models:
           

---
## 2. Initialize Models

All models in teradatagenai collection are of type `TeradataAI`. Here we'll initialize both an embedding model and a chat completion model using the hosted models with the 'td_hosted' api_type.

### Initialize Embedding Model

In [35]:
hosted_embedding_model = TeradataAI(api_type = "td_hosted",
                                    model_name = "amazon.titan-embed-text-v2:0")


hosted_chat_model = TeradataAI(api_type = "td_hosted",
                                model_name = "anthropic.claude-3-haiku-20240307-v1:0",
                                )

print("✓ Embedding and Chat models initialized")

✓ Embedding and Chat models initialized


In [ ]:
# Set AWS credentials in environment
os.environ['AWS_ACCESS_KEY_ID'] = '<Enter AWS access key>'
os.environ['AWS_SECRET_ACCESS_KEY'] = '<Enter AWS secret key>'
os.environ['AWS_DEFAULT_REGION'] = '<Enter your region>'


# Configure the AWS Bedrock embedding model
aws_provider_chat_model = TeradataAI(
    api_type="aws",
    model_name="anthropic.claude-3-5-sonnet-20240620-v1:0"
)

---
## 3. Create Content-Based Collection

Now we'll create a content-based collection using the `create()` method. 

**Step 1**: Define column information using `ColumnInfo`  
**Step 2**: Build a `ContentBasedIndex` with the column definitions  
**Step 3**: Initialize a Collection instance  
**Step 4**: Call `create()` with the index

We'll configure:
- **key_columns**: Unique identifiers for each record
- **data_columns**: Text content to embed
- **metadata_columns**: Additional information to store
- **search_params**: HNSW algorithm configuration

In [9]:
# Step 1: Create DataFrame reference to the data
reviews_df = DataFrame(in_schema("vs_user", "amazon_reviews_25"))
print("✓ DataFrame created")

✓ DataFrame created


In [10]:
reviews_df.head(1)

rev_id,aid,rev_name,helpful,rev_text,rating,prodsummary,unixrevtime,revtime
A26GKZPS079GFF,000100039X,Areej,"[2, 3]","I would have to say that this is the best book I""ve ever read.. I could feel every word deep in my heart everytime, of the many times I""ve read it! I would never get enough of it! its a treasure..",5.000,Touches my heart.. again and.. again...,982972800,"02 24, 2001"


In [11]:
# Step 2: Define column information using ColumnInfo
rev_id_column = ColumnInfo(
    name="rev_id",
    datatype=VARCHAR(50),
    description="Review ID - Primary Key",
    sources=[reviews_df.rev_id]
)

aid_column = ColumnInfo(
    name="aid",
    datatype=VARCHAR(50),
    description="Author ID - Primary Key",
    sources=[reviews_df.aid]
)

rev_text_column = ColumnInfo(
    name="rev_text",
    datatype=VARCHAR(64000),
    description="Review text for embeddings",
    sources=[reviews_df.rev_text]
)

prodsummary_column = ColumnInfo(
    name="prodsummary",
    datatype=VARCHAR(4000),
    description="Product summary metadata",
    sources=[reviews_df.prodsummary]
)

rev_name_column = ColumnInfo(
    name="rev_name",
    datatype=VARCHAR(500),
    description="Review name metadata",
    sources=[reviews_df.rev_name]
)

print("✓ Column definitions created")

✓ Column definitions created


In [12]:
# Step 3: Build the ContentBasedIndex using the DataFrame
content_index = ContentBasedIndex(
    object_names=reviews_df,
    key_columns=[rev_id_column, aid_column],
    data_columns=[rev_text_column],
    metadata_columns=[prodsummary_column, rev_name_column]
)

print("✓ ContentBasedIndex created")

✓ ContentBasedIndex created


In [13]:
# Step 4: Define HNSW parameters for indexing algorithm
hnsw_params = HNSW(
    metric = "COSINE",
    ef_construction=15,
    ef=100
)

print("✓ HNSW parameters defined")

✓ HNSW parameters defined


In [14]:
# Step 5: Initialize Collection and create
collection = Collection(name='collection_amazon_reviews')

collection.create(
    type=CollectionType.CONTENT_BASED,
    index=content_index,
    embedding_model=hosted_embedding_model,
    chat_model=hosted_chat_model,
    indexing_algorithm=hnsw_params
)

print(f"✓ Collection '{collection.name}' created")

/root/TDGENAI-RC-20.0.0.6/tdg-rc/lib/python3.12/site-packages/teradatagenai/vector_store/collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
✓ Collection 'collection_amazon_reviews' created


### Check Creation Status

In [15]:
print("\nCollection Status:")
collection.status()


Collection Status:


collection_name,collection_status
collection_amazon_reviews,ready


---
## 4. Retrieve Collection Information

Let's explore various methods to get information about our collection.

### Get Collection Details

In [16]:
print("Collection Details:")
collection.get_details()

Collection Details:


collection_name,collection_status,collection_type,collection_description,target_database,permission,last_updated
collection_amazon_reviews,ready,CONTENT-BASED,None,vs_user,ADMIN,2026-02-19 12:36:37.772670+00:00


An elaborate version of collection details can be retrieved as well

In [17]:
collection.get_details(elaborate = True, return_type = "json")

{'collection_name': 'collection_amazon_reviews',
 'collection_status': 'ready',
 'collection_type': 'CONTENT-BASED',
 'collection_description': None,
 'target_database': 'vs_user',
 'permission': 'ADMIN',
 'last_updated': '2026-02-19 12:36:37.772670+00:00',
 'error_message': None,
 'embedding_datatype': 'VECTOR32',
 'use_simd': False,
 'object_names': ['vs_user.amazon_reviews_25'],
 'key_columns': {'vs_user.amazon_reviews_25': ['rev_id', 'aid']},
 'key_columns_info': [{'name': 'rev_id',
   'datatype': 'VARCHAR(50)',
   'description': 'Review ID - Primary Key'},
  {'name': 'aid',
   'datatype': 'VARCHAR(50)',
   'description': 'Author ID - Primary Key'}],
 'metadata_columns': {'vs_user.amazon_reviews_25': ['prodsummary',
   'rev_name']},
 'metadata_columns_info': [{'name': 'prodsummary',
   'datatype': 'VARCHAR(4000)',
   'description': 'Product summary metadata'},
  {'name': 'rev_name',
   'datatype': 'VARCHAR(500)',
   'description': 'Review name metadata'}],
 'data_columns': {'vs_use

### Get Model Information

The `get_model_info()` method returns information about the search model (HNSW in this case).

In [18]:
print("\nModel Information:")
collection.get_model_info()


Model Information:


{'hnsw_model':                                                                                                                   TD_HNSW_Vect32_Nodes
 0  b'-16410AF93882DC0FDFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFF3FFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 1  b'-16410AF93882DC0FEFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFECFFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 2  b'-16410AF93882DC0FEFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFE8FFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 3  b'-16410AF93882DC0FEFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFE9FFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 4  b'-16410AF93882DC0FEFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFE4FFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 5  b'-16410AF93882DC0FEFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFF9FFFFFFF0FFFFFFE1FFFFFFFFFFFFFFFFFFFFFFDFFFFFFFDFFFFFFFFDFFFFFFFFFBFF0000000000'
 6  b'-16410AF93882DC0FEFFFFFFFFFF

### Get Indexes and Embeddings

This method returns the underlying vector store table containing embeddings and metadata.

In [19]:
indexes_embeddings = collection.get_indexes_embeddings()
print("\nIndexes and Embeddings:")
print(f"Shape: {indexes_embeddings.shape}")
indexes_embeddings.head(1)


Indexes and Embeddings:
Shape: (25, 10)


DataBaseName,TableName,TD_ID,rev_id,aid,prodsummary,rev_name,rev_text,vector_index,vector_index_normalized
vs_user,amazon_reviews_25,23,A3FI0744PG1WYG,000100039X,The Prophet,"""Always Reading """"tkm""""""","This is a timeless classic. Over the years I""ve given it as a gift more times than I can count, and will continue to do so. Addresses real life issues in a beautiful way and makes us reexamine our own attitude about how we see what happens in our lives. So easy to read over and over.","-0.0506170988082886,0.0380402654409409,-0.0298719294369221,-0.0413605533540249,0.0344944708049297,-0.0470124073326588,0.0433178842067719,0.00446484424173832,0.0621440894901752,-0.0372500978410244,0.0233138892799616,0.0195373725146055,0.0580349490046501,-0.0192663818597794,-0.0884115621447563,-0.00823645666241646,-0.00254231155849993,-0.0407746657729149,-0.0425436198711395,0.0567450150847435,-0.00477821053937078,-0.00322733027860522,0.0311735793948174,-0.00591034488752484,-0.00660885544493794,0.0299691427499056,0.0193183291703463,-0.00905009265989065,0.0303137507289648,-0.0303059034049511,-0.000399952870793641,0.0434350185096264,0.0238276347517967,-0.0115130012854934,-0.0146134234964848,0.0589477121829987,0.0136636905372143,0.0307998042553663,0.00124618934933096,-0.0149464253336191,0.0241050124168396,0.0021372502669692,-0.00272228685207665,0.0152629874646664,0.0588056556880474,0.0321239680051804,0.000962144986260682,0.018332539126277,0.0510467551648617,0.0331953316926956,0.0123577900230885,-0.0247417334467173,","-0.0506170988082886,0.0380402654409409,-0.0298719294369221,-0.0413605533540249,0.0344944708049297,-0.0470124073326588,0.0433178842067719,0.00446484424173832,0.0621440894901752,-0.0372500978410244,0.0233138892799616,0.0195373725146055,0.0580349490046501,-0.0192663818597794,-0.0884115621447563,-0.00823645666241646,-0.00254231155849993,-0.0407746657729149,-0.0425436198711395,0.0567450150847435,-0.00477821053937078,-0.00322733027860522,0.0311735793948174,-0.00591034488752484,-0.00660885544493794,0.0299691427499056,0.0193183291703463,-0.00905009265989065,0.0303137507289648,-0.0303059034049511,-0.000399952870793641,0.0434350185096264,0.0238276347517967,-0.0115130012854934,-0.0146134234964848,0.0589477121829987,0.0136636905372143,0.0307998042553663,0.00124618934933096,-0.0149464253336191,0.0241050124168396,0.0021372502669692,-0.00272228685207665,0.0152629874646664,0.0588056556880474,0.0321239680051804,0.000962144986260682,0.018332539126277,0.0510467551648617,0.0331953316926956,0.0123577900230885,-0.0247417334467173,"


---
## 5. Update Collection

Collections can be updated in various ways. Let's demonstrate:
<br/> 1. Adding datasets
<br/> 2. Deleting datasets


### Adding Data

We'll add the `amazon_reviews_10` dataset to our existing collection using the `update()` method with a new ContentBasedIndex.

In [20]:
# Create DataFrame for additional data
additional_reviews_df = DataFrame(in_schema("vs_user", "amazon_reviews_10"))

# Create index for additional data
additional_data_index = ContentBasedIndex(object_names=additional_reviews_df)

# Add the dataset to collection using update()
collection.update(
    index=additional_data_index,
    alter_operation='ADD',
    update_style='MINOR'
)

print("✓ Dataset added to collection")

Collection update request is accepted and in progress
✓ Dataset added to collection


In [23]:
# Check status after adding
print("\nAdd Dataset Status:")
collection.status()


Add Dataset Status:


collection_name,collection_status
collection_amazon_reviews,ready


### Deleting Data

Now we'll remove the dataset we just added using the `update()` method.

In [24]:
collection.update(
    index=additional_data_index,
    alter_operation='DELETE',
    update_style='MINOR'
)
print("✓ Dataset deleted from collection")

Collection update request is accepted and in progress
✓ Dataset deleted from collection


In [26]:
# Check status after adding
print("Delete Dataset Status:")
collection.status()

Delete Dataset Status:


collection_name,collection_status
collection_amazon_reviews,ready


---
## 6. Search Operations

Now let's perform various search operations on our collection.

### Similarity Search

Find the most similar documents based on a question.

In [27]:
question = 'Positive reviews'

search_results = collection.similarity_search(
    question=question,
    top_k=10
)

print(search_results)

similar_objects_count:10
similar_objects:
      score DataBaseName          TableName  TD_ID          rev_id         aid                                         prodsummary                       rev_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

### Prepare Response

Generate a natural language response based on similarity search results.

In [38]:
question = 'What are customers saying about Gibrans books?'
prompt = 'Provide a concise summary of customer feedback. Format the response in a conversational way.'

# First perform similarity search
search_results = collection.similarity_search(
    question=question,
    top_k=5
)

# Then prepare response based on results
response = collection.prepare_response(
    question=question,
    similarity_results=search_results,
    prompt=prompt,
    hosted_embedding_model=hosted_embedding_model,
    chat_model=aws_provider_chat_model
)

print(f"\nQuestion: {question}\n")
print("Response:")
print(response)


Question: What are customers saying about Gibrans books?

Response:
Based on the customer feedback provided, readers are overwhelmingly positive about Gibran's books, particularly "The Prophet." Here's a summary of what customers are saying:

1. Many readers find Gibran's work deeply inspiring and spiritually enriching. One customer described it as bringing "spiritual and visual beauty to life within you."

2. The writing style is highly praised. Customers appreciate Gibran's poetic style, use of metaphors, and the almost biblical feel of his work.

3. Readers find the content profound yet simple, describing it as "honest," "moving," and "inspirational."

4. Several customers mention that Gibran's work, especially "The Prophet," is a "must-read" and a "timeless classic."

5. The book is seen as a guide for life decisions and personal growth. One reader suggested it could be more helpful than self-help programs or therapy.

6. Customers appreciate the universal appeal of Gibran's writi

### Ask (Combined Search + Response)

The `ask()` method combines similarity search and response generation in a single call for convenience.

In [41]:
custom_prompt = """List reviews about the books. Only provide information present in the data.
Format results like this:
- Review ID: 
- Product Summary: 
- Review: 
"""

question = 'Are there any reviews saying that the books are inspiring?'

response = collection.ask(
    question=question,
    prompt=custom_prompt,
    top_k=5,
    chat_model = hosted_chat_model
)

print(f"\nQuestion: {question}\n")
print("Response:")
print(response)


Question: Are there any reviews saying that the books are inspiring?

Response:
Yes, there are a few reviews that mention the books being inspiring:

- Review ID: 0
Product Summary: Wonderful!
Review: Spiritually and mentally inspiring! A book that allows you to question your morals and will help you discover who you really are!

- Review ID: 1
Product Summary: A book everyone "should" read
Review: If you were looking for an example of an intelligently designed (pun intended) book of spiritual guidance, this would be it. It just cuts to the heart of how to live and how to relate to others. If Jesus and the Buddha teamed up to write a book, it might come out like this.


---
## 8. Cleanup

### Destroy the Collection

When you're done, destroy the collection to free up resources.

In [ ]:
collection.destroy()

---
## 7. Alternative Methods: Dataset Management

In addition to the `update()` method, teradatagenai provides convenient methods specifically designed for dataset management:

- **`from_datasets()`**: Create a collection directly from datasets
- **`add_datasets()`**: Add tables/DataFrames to an existing collection
- **`delete_datasets()`**: Remove tables/DataFrames from a collection

These methods provide a more intuitive API when working specifically with dataset operations.

### Create Collection with from_datasets()

The `from_datasets()` class method provides a streamlined way to create a collection directly from datasets.

In [48]:
data_amazon_25 = DataFrame(in_schema("vs_user", "amazon_reviews_25"))
data_amazon_10 = DataFrame(in_schema("vs_user", "amazon_reviews_10"))

In [ ]:
# Create the collection using from_datasets
collection_alt = Collection.from_datasets(
    name="collection_content_based_alt",
    data= data_amazon_25,
    embedding=hosted_embedding_model,
    key_columns=rev_id_column,
    data_columns=rev_text_column,
    description="Content-based collection created from amazon reviews"
)

print(f"✓ Collection '{collection_alt.name}' created using from_datasets()")
collection_alt.status()

/root/TDGENAI-RC-20.0.0.6/tdg-rc/lib/python3.12/site-packages/teradatagenai/vector_store/collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
✓ Collection 'collection_content_based_alt' created using from_datasets()


collection_name,collection_status,retry_after
collection_content_based_alt,create_started,60


In [51]:
collection_alt.status()

collection_name,collection_status
collection_content_based_alt,ready


### Add Datasets with add_datasets()

Use `add_datasets()` to add more tables/DataFrames to an existing collection. This method can also create a new collection if it doesn't exist.

In [49]:
# Add additional data to the collection
collection_alt.add_datasets(
    data = data_amazon_10,
    update_style="MINOR"
)

print("✓ Dataset added using add_datasets()")
collection_alt.status()

Collection update request is accepted and in progress
✓ Dataset added using add_datasets()


collection_name,collection_status,retry_after
collection_content_based_alt,update_started,60


### Delete Datasets with delete_datasets()

Remove specific tables/DataFrames from an existing collection using `delete_datasets()`.

In [52]:
# Delete the dataset we added earlier
collection_alt.delete_datasets(
    data="amazon_reviews_10"
)

print("✓ Dataset deleted using delete_datasets()")
collection_alt.status()

Collection update request is accepted and in progress
✓ Dataset deleted using delete_datasets()


collection_name,collection_status,retry_after
collection_content_based_alt,update_started,60


In [53]:
collection_alt.status()

collection_name,collection_status
collection_content_based_alt,ready


### Cleanup Alternative Collections

Let's clean up the collections created in this section.

In [ ]:
# Destroy the collections created in this section
collection_alt.destroy()

print("✓ Alternative collections destroyed")

In [42]:
collection.destroy()
print(f"✓ Collection '{collection.name}' destroyed")

Collection destroy request is accepted and in progress
✓ Collection 'collection_amazon_reviews' destroyed


### Remove Database Connection

In [32]:
remove_context()
print("✓ Database connection removed")

✓ Database connection removed


---
## Summary

In this notebook, you learned how to:

✅ Set up your environment and connect to Teradata Vantage  
✅ Initialize embedding and chat models using TeradataAI  
✅ Create a content-based collection from datasets  
✅ Retrieve collection details, model info, and embeddings  
✅ Update collections by modifying parameters and adding/deleting datasets  
✅ Perform similarity search operations  
✅ Generate natural language responses using `prepare_response()` and `ask()`  
✅ Use alternative dataset management methods (`from_datasets()`, `add_datasets()`, `delete_datasets()`)  
✅ Clean up resources properly  
